# Per-start dataset (2021–2026)

One row per starting pitcher appearance. Targets are the starter's hits, strikeouts, walks and earned runs in that game. Features only use games from earlier dates.

Pulls box scores from the MLB Stats API (`statsapi.mlb.com`), saves gzipped raw CSVs to `data/raw/`, then builds `data/starts_dataset.csv.gz`.

Column prefixes: `p_` pitcher form, `lu_` lineup averages, `b1_`–`b9_` each opposing batter by slot (for the lineup-aware model).

In [1]:
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import requests

API = "https://statsapi.mlb.com/api/v1"
SEASONS = range(2021, 2027)
TRAIN_SEASONS = [2021, 2022, 2023, 2024]
DATA = Path("data")
RAW = DATA / "raw"
RAW.mkdir(parents=True, exist_ok=True)

session = requests.Session()

## Pull box scores

In [2]:
def get(url, params=None, tries=5):
    for i in range(tries):
        try:
            r = session.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if i == tries - 1:
                raise
            time.sleep(2 ** i)


def schedule(season):
    data = get(f"{API}/schedule", {"sportId": 1, "season": season, "gameType": "R"})
    games = {}
    for d in data["dates"]:
        for g in d["games"]:
            if g["status"]["codedGameState"] == "F":
                games[g["gamePk"]] = d["date"]
    return games


def parse_game(game_pk, date):
    box = get(f"{API}/game/{game_pk}/boxscore")
    starts, bats = [], []
    for side, opp in [("home", "away"), ("away", "home")]:
        team = box["teams"][side]
        base = {"game_pk": game_pk, "date": date, "team_id": team["team"]["id"],
                "opp_id": box["teams"][opp]["team"]["id"], "home": int(side == "home")}

        # first pitcher listed is the starter
        if team["pitchers"]:
            p = team["players"][f"ID{team['pitchers'][0]}"]
            s = p["stats"]["pitching"]
            starts.append({**base, "pitcher_id": p["person"]["id"],
                           "outs": s.get("outs", 0), "bf": s.get("battersFaced", 0),
                           "pitches": s.get("numberOfPitches", 0),
                           "h": s.get("hits", 0), "k": s.get("strikeOuts", 0),
                           "bb": s.get("baseOnBalls", 0), "er": s.get("earnedRuns", 0),
                           "hr": s.get("homeRuns", 0), "hbp": s.get("hitByPitch", 0)})

        for p in team["players"].values():
            if "battingOrder" not in p:
                continue
            s = p["stats"].get("batting", {})
            order = int(p["battingOrder"])
            bats.append({**base, "batter_id": p["person"]["id"],
                         # 100, 200, ... 900 = starting lineup slot; 101 etc = sub
                         "slot": order // 100 if order % 100 == 0 else 0,
                         "pa": s.get("plateAppearances", 0), "ab": s.get("atBats", 0),
                         "h": s.get("hits", 0), "2b": s.get("doubles", 0),
                         "3b": s.get("triples", 0), "hr": s.get("homeRuns", 0),
                         "bb": s.get("baseOnBalls", 0), "k": s.get("strikeOuts", 0),
                         "hbp": s.get("hitByPitch", 0), "sf": s.get("sacFlies", 0)})
    return starts, bats


def pull_season(season):
    games = schedule(season)
    starts, bats = [], []
    with ThreadPoolExecutor(8) as pool:
        for i, (s, b) in enumerate(pool.map(lambda kv: parse_game(*kv), games.items())):
            starts += s
            bats += b
            if i % 500 == 0:
                print(season, i, "/", len(games), flush=True)
    pd.DataFrame(starts).assign(season=season).to_csv(RAW / f"starts_{season}.csv.gz", index=False)
    pd.DataFrame(bats).assign(season=season).to_csv(RAW / f"batting_{season}.csv.gz", index=False)

In [3]:
# skips seasons already saved; takes ~3 min per season
for season in SEASONS:
    if not (RAW / f"starts_{season}.csv.gz").exists():
        pull_season(season)

In [4]:
def pull_people():
    ids = set()
    for f in RAW.glob("starts_*.csv.gz"):
        ids |= set(pd.read_csv(f)["pitcher_id"])
    for f in RAW.glob("batting_*.csv.gz"):
        ids |= set(pd.read_csv(f)["batter_id"])
    ids = sorted(ids)
    rows = []
    for i in range(0, len(ids), 200):
        chunk = ",".join(map(str, ids[i:i + 200]))
        for p in get(f"{API}/people", {"personIds": chunk})["people"]:
            rows.append({"player_id": p["id"], "name": p["fullName"],
                         "throws": p.get("pitchHand", {}).get("code"),
                         "bats": p.get("batSide", {}).get("code")})
    pd.DataFrame(rows).to_csv(RAW / "players.csv.gz", index=False)


if not (RAW / "players.csv.gz").exists():
    pull_people()

## Build features

In [5]:
SHRINK = 100  # pseudo PA/BF to pull small samples toward league average
LAST_N = 5


def load(prefix):
    df = pd.concat([pd.read_csv(f) for f in sorted(RAW.glob(f"{prefix}_*.csv.gz"))])
    df["date"] = pd.to_datetime(df["date"])
    return df


def shrink(num, den, prior):
    return (num + prior * SHRINK) / (den + SHRINK)


def window_sums(df, key, cols):
    # sums over the previous 365 days, excluding the current date
    daily = df.groupby([key, "date"])[cols].sum().reset_index().sort_values([key, "date"])
    return (daily.set_index("date").groupby(key)[cols]
            .rolling("365D", closed="left").sum().fillna(0).reset_index())

In [6]:
def pitcher_features(starts, throws):
    s = starts.sort_values(["pitcher_id", "date", "game_pk"]).reset_index(drop=True)
    train = s[s.season.isin(TRAIN_SEASONS)]
    per_start = train[["h", "k", "bb", "er", "outs", "pitches"]].mean()
    lg = train[["h", "k", "bb", "er", "hr", "bf", "outs"]].sum()

    g = s.groupby("pitcher_id")
    for c in ["h", "k", "bb", "er", "outs", "pitches"]:
        s[f"p_l{LAST_N}_{c}"] = g[c].transform(
            lambda x: x.shift(1).rolling(LAST_N, min_periods=1).mean()).fillna(per_start[c])
    s["p_rest_days"] = g["date"].diff().dt.days.clip(upper=30).fillna(30)

    s["n"] = 1
    r = window_sums(s, "pitcher_id", ["n", "bf", "outs", "h", "k", "bb", "er", "hr"])
    s = s.merge(r, on=["pitcher_id", "date"], suffixes=("", "_365"))
    s = s.sort_values(["pitcher_id", "date", "game_pk"]).reset_index(drop=True)
    s["p_n_starts"] = s["n_365"]
    for c in ["k", "bb", "h", "hr"]:
        s[f"p_{c}_rate"] = shrink(s[f"{c}_365"], s["bf_365"], lg[c] / lg["bf"])
    s["p_er9"] = 27 * shrink(s["er_365"], s["outs_365"], lg["er"] / lg["outs"])
    s["p_outs_per_start"] = (s["outs_365"] + 3 * per_start["outs"]) / (s["n_365"] + 3)

    # season-to-date per-start averages (for the average baseline)
    gs = s.groupby(["pitcher_id", "season"])
    s["p_std_n"] = gs.cumcount()
    for c in ["h", "k", "bb", "er"]:
        prev = gs[c].cumsum() - s[c]
        s[f"p_std_{c}"] = np.where(s["p_std_n"] > 0, prev / s["p_std_n"].clip(lower=1),
                                   s[f"p_l{LAST_N}_{c}"])

    s = s.merge(throws, left_on="pitcher_id", right_on="player_id", how="left")
    s["p_throws_R"] = (s["throws"] == "R").astype(int)
    return s.drop(columns=[c for c in s.columns if c.endswith("_365")] + ["n", "player_id", "throws"])


def batter_features(bat):
    cols = ["pa", "ab", "h", "2b", "3b", "hr", "bb", "k", "hbp", "sf"]
    t = bat[bat.season.isin(TRAIN_SEASONS)][cols].sum()
    r = window_sums(bat, "batter_id", cols)
    out = r[["batter_id", "date", "pa"]].copy()
    out["k_rate"] = shrink(r["k"], r["pa"], t["k"] / t["pa"])
    out["bb_rate"] = shrink(r["bb"], r["pa"], t["bb"] / t["pa"])
    out["obp"] = shrink(r["h"] + r["bb"] + r["hbp"], r["pa"], (t["h"] + t["bb"] + t["hbp"]) / t["pa"])
    out["iso"] = shrink(r["2b"] + 2 * r["3b"] + 3 * r["hr"], r["ab"],
                        (t["2b"] + 2 * t["3b"] + 3 * t["hr"]) / t["ab"])
    return out

In [7]:
starts = load("starts")
bat = load("batting")
players = pd.read_csv(RAW / "players.csv.gz")

df = pitcher_features(starts, players[["player_id", "throws"]])

lineup = bat[bat.slot > 0][["game_pk", "team_id", "batter_id", "date", "slot"]]
lineup = lineup.merge(batter_features(bat), on=["batter_id", "date"], how="left")
lineup = lineup.merge(players[["player_id", "bats"]], left_on="batter_id",
                      right_on="player_id", how="left").drop(columns="player_id")

# only keep games with a full 1-9 lineup
full = lineup.groupby(["game_pk", "team_id"])["slot"].nunique() == 9
lineup = lineup.set_index(["game_pk", "team_id"]).loc[full[full].index].reset_index()

df = df.merge(lineup[["game_pk", "team_id", "slot", "batter_id", "bats", "pa", "k_rate",
                      "bb_rate", "obp", "iso"]],
              left_on=["game_pk", "opp_id"], right_on=["game_pk", "team_id"],
              suffixes=("", "_b"))
df["platoon_adv"] = ((df["bats"] == "S") |
                     ((df["bats"] == "L") & (df["p_throws_R"] == 1)) |
                     ((df["bats"] == "R") & (df["p_throws_R"] == 0))).astype(int)
df["bats_L"] = (df["bats"] == "L").astype(int)

In [8]:
feats = ["pa", "k_rate", "bb_rate", "obp", "iso", "platoon_adv", "bats_L"]
key = ["game_pk", "pitcher_id"]

wide = df.pivot(index=key, columns="slot", values=feats)
wide.columns = [f"b{slot}_{f}" for f, slot in wide.columns]
agg = df.groupby(key)[feats].mean().add_prefix("lu_")
ids = df.pivot(index=key, columns="slot", values="batter_id").add_prefix("b").add_suffix("_id")

base = df.drop_duplicates(key).set_index(key)
targets = base[["h", "k", "bb", "er"]].rename(
    columns={"h": "hits", "k": "strikeouts", "bb": "walks", "er": "earned_runs"})
p_cols = [c for c in base.columns if c.startswith("p_")] + ["home"]

dataset = pd.concat([base[["date", "season", "team_id", "opp_id"]], targets,
                     base[p_cols], agg, wide, ids], axis=1).reset_index()
dataset = dataset.sort_values(["date", "game_pk"]).reset_index(drop=True)
dataset.to_csv(DATA / "starts_dataset.csv.gz", index=False)

print(f"{len(dataset)} starts, {dataset.shape[1]} columns")
dataset.groupby("season").size()

29154 starts, 110 columns


season
2021    4858
2022    4860
2023    4860
2024    4858
2025    4860
2026    4858
dtype: int64

## Checks

In [9]:
print("missing values:", dataset.isna().sum().sum())
print("duplicate starts:", dataset.duplicated(["game_pk", "pitcher_id"]).sum())
dataset[["hits", "strikeouts", "walks", "earned_runs"]].describe().round(2)

missing values: 0
duplicate starts: 0


,hits,strikeouts,walks,earned_runs
count,29154.00,29154.00,29154.00,29154.00
mean,4.87,4.78,1.70,2.42
std,2.26,2.52,1.29,1.98
min,0.00,0.00,0.00,0.00
25%,3.00,3.00,1.00,1.00
50%,5.00,5.00,2.00,2.00
75%,6.00,6.00,2.00,4.00
max,14.00,16.00,8.00,13.00
